# Authentication Anomaly Detection

Build an explainable model that ranks suspicious sign-in events using identity and access telemetry.

**Safety and scope:** This project uses synthetic, non-sensitive telemetry for defensive analytics. It does not perform exploitation or execute malicious content.

## Goal

Train a logistic-risk model, evaluate its detection quality, and inspect the highest-risk login events.


## Setup

The notebook is deterministic, runs offline, and implements the core analytical method directly with NumPy and Pandas so the modeling logic remains inspectable.


In [1]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

def sigmoid(values):
    clipped = np.clip(values, -30, 30)
    return 1.0 / (1.0 + np.exp(-clipped))

def split_indices(size, test_fraction=0.25):
    shuffled = rng.permutation(size)
    split_at = int(size * (1 - test_fraction))
    return shuffled[:split_at], shuffled[split_at:]

def standardize(train_values, test_values):
    mean = train_values.mean(axis=0)
    std = train_values.std(axis=0)
    std = np.where(std < 1e-9, 1.0, std)
    return (train_values - mean) / std, (test_values - mean) / std, mean, std

def fit_logistic(features, labels, steps=1400, learning_rate=0.08, l2=0.01):
    design = np.column_stack([np.ones(len(features)), features])
    weights = np.zeros(design.shape[1])
    for _ in range(steps):
        probabilities = sigmoid(design @ weights)
        gradient = design.T @ (probabilities - labels) / len(labels)
        gradient[1:] += l2 * weights[1:]
        weights -= learning_rate * gradient
    return weights

def predict_probability(features, weights):
    design = np.column_stack([np.ones(len(features)), features])
    return sigmoid(design @ weights)

def classification_metrics(labels, predictions):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    tp = int(((labels == 1) & (predictions == 1)).sum())
    tn = int(((labels == 0) & (predictions == 0)).sum())
    fp = int(((labels == 0) & (predictions == 1)).sum())
    fn = int(((labels == 1) & (predictions == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    accuracy = (tp + tn) / max(len(labels), 1)
    return pd.Series({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    })


## Steps

### 1. Generate synthetic authentication telemetry


In [2]:
event_count = 1800
failed_attempts = rng.poisson(0.7, event_count)
new_device = rng.binomial(1, 0.14, event_count)
country_mismatch = rng.binomial(1, 0.07, event_count)
impossible_travel = rng.binomial(1, 0.035, event_count)
off_hours = rng.binomial(1, 0.24, event_count)
privileged_user = rng.binomial(1, 0.10, event_count)
source_reputation = rng.beta(1.4, 5.0, event_count)

compromise_probability = sigmoid(
    -5.1
    + 0.65 * failed_attempts
    + 1.20 * new_device
    + 1.75 * country_mismatch
    + 2.20 * impossible_travel
    + 0.75 * off_hours
    + 0.90 * privileged_user
    + 3.20 * source_reputation
)
compromised = rng.binomial(1, compromise_probability)

auth_events = pd.DataFrame({
    "failed_attempts": failed_attempts,
    "new_device": new_device,
    "country_mismatch": country_mismatch,
    "impossible_travel": impossible_travel,
    "off_hours": off_hours,
    "privileged_user": privileged_user,
    "source_reputation": source_reputation.round(3),
    "compromised": compromised,
})

print("Dataset shape:", auth_events.shape)
print("Compromise rate:", round(auth_events["compromised"].mean(), 3))
print(auth_events.head(6).to_string(index=False))


Dataset shape: (1800, 8)
Compromise rate: 0.063
 failed_attempts  new_device  country_mismatch  impossible_travel  off_hours  privileged_user  source_reputation  compromised
               1           1                 0                  0          0                0              0.269            0
               2           0                 0                  0          1                0              0.557            0
               3           1                 0                  1          0                0              0.085            0
               0           0                 0                  0          0                0              0.006            0
               0           0                 0                  0          0                0              0.158            0
               2           0                 0                  0          0                0              0.108            0


### 2. Train and explain the model


In [3]:
feature_names = [column for column in auth_events.columns if column != "compromised"]
train_index, test_index = split_indices(len(auth_events))
train_features = auth_events.loc[train_index, feature_names].to_numpy(float)
test_features = auth_events.loc[test_index, feature_names].to_numpy(float)
train_labels = auth_events.loc[train_index, "compromised"].to_numpy(int)
test_labels = auth_events.loc[test_index, "compromised"].to_numpy(int)

train_scaled, test_scaled, feature_mean, feature_std = standardize(train_features, test_features)
weights = fit_logistic(train_scaled, train_labels)
test_probability = predict_probability(test_scaled, weights)
decision_threshold = 0.06
test_prediction = (test_probability >= decision_threshold).astype(int)
auth_metrics = classification_metrics(test_labels, test_prediction)

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "standardized_weight": weights[1:],
}).sort_values("standardized_weight", ascending=False)

ranked_events = auth_events.loc[test_index].copy()
ranked_events["risk_probability"] = test_probability
ranked_events = ranked_events.sort_values("risk_probability", ascending=False)

print("Test metrics:")
print(auth_metrics.round(3).to_string())
print("\nMost influential risk features:")
print(coefficient_table.head(7).round(3).to_string(index=False))
print("\nHighest-risk sign-ins:")
print(ranked_events.head(8).round(3).to_string(index=False))


Test metrics:
accuracy       0.687
precision      0.145
recall         0.821
f1             0.246
tp            23.000
fp           136.000
tn           286.000
fn             5.000

Most influential risk features:
          feature  standardized_weight
  failed_attempts                0.378
 country_mismatch                0.372
        off_hours                0.352
       new_device                0.350
impossible_travel                0.332
source_reputation                0.312
  privileged_user                0.164

Highest-risk sign-ins:
 failed_attempts  new_device  country_mismatch  impossible_travel  off_hours  privileged_user  source_reputation  compromised  risk_probability
               5           1                 0                  1          0                0              0.345            1             0.809
               2           0                 0                  1          1                0              0.317            1             0.467
               2 

## Checks


In [4]:
assert not auth_events.isna().any().any()
assert 0.01 < auth_events["compromised"].mean() < 0.50
assert auth_metrics["recall"] >= 0.55
assert ranked_events["risk_probability"].is_monotonic_decreasing
print("Checks passed: complete data, plausible class balance, usable recall, and sorted risk queue.")


Checks passed: complete data, plausible class balance, usable recall, and sorted risk queue.


## Next Steps

        - Replace the generator with identity-provider sign-in logs and document the event schema.
- Add user and peer-group baselines to reduce false positives.
- Calibrate the decision threshold against analyst capacity and incident cost.
